# Job Market Intelligence & Resume Matcher Prototype
This notebook demonstrates parsing raw job descriptions with `JobExtractor`, storing and querying records in a local DuckDB data warehouse, and running robust resume skill-matching analysis using `JobMatcher`.

In [1]:
import sys
from pathlib import Path
import pandas as pd

# Ensure project root is in path for local src imports
sys.path.append(str(Path('..').resolve()))

from src.processing.extractor import JobExtractor
from src.processing.matcher import JobMatcher
from src.database.connection import get_connection, init_db

# 1. Sample raw job description text (simulating a scraped page)
sample_job = """
    We are looking for a Junior Python Developer based in Kuching.
    Requirements:
    - 1+ years of experience with Python, FastAPI, and SQL.
    - Familiarity with Docker and AWS/Cloud certifications is a huge plus.
    - Salary: RM 3,500 - RM 5,000 per month.
"""

# 2. Initialize modular JobExtractor
extractor = JobExtractor()
parsed_data = extractor.parse_job(sample_job)
print("Extracted Data:", parsed_data)

# 3. Initialize local DuckDB warehouse connection & schema
init_db()
con = get_connection(read_only=False)

# Insert the parsed sample data
con.execute("""
    INSERT INTO job_listings (id, title, skills, salary_range)
    VALUES (1, 'Junior Python Developer', ?, ?)
""", [parsed_data['skills'], parsed_data['salary_range']])

# 4. Demonstrate Resume Skill Matching (with safe list casting)
user_skills = ['Python', 'SQL', 'Docker', 'FastAPI']
job_skills_from_db = parsed_data['skills']

clean_user_skills = list(user_skills)
clean_job_skills = list(job_skills_from_db) if job_skills_from_db is not None else []

match_result = JobMatcher.calculate_match(clean_user_skills, clean_job_skills)
print(f"\nResume Match Score: {match_result['match_percentage']}%")
print(f"Matching Skills: {match_result['matching_skills']}")
print(f"Missing Skills: {match_result['missing_skills']}")

# 5. Query back to verify warehouse
df_result = con.execute("SELECT id, title, skills, salary_range, created_at FROM job_listings").fetchdf()
print("\nDuckDB Query Result:")
display(df_result)

con.close()

Extracted Data: {'skills': ['AWS', 'FASTAPI', 'PYTHON', 'CLOUD', 'SQL', 'DOCKER'], 'salary_range': 'RM 3,500 - RM 5,000'}

Resume Match Score: 66.67%
Matching Skills: ['DOCKER', 'FASTAPI', 'PYTHON', 'SQL']
Missing Skills: ['AWS', 'CLOUD']

DuckDB Query Result:


,id,title,skills,salary_range,created_at
0,1,Junior Python Developer (Kuching),"[PYTHON, SQL, FASTAPI, DOCKER]","RM 3,800 - RM 5,500",2026-09-12 18:46:20.489587
1,2,Marketing Student Assistant,[GO],"RM,",2026-09-12 18:46:21.359973
2,3,Engineering Manager TLM Platform,"[DOCKER, AWS, GCP, KUBERNETES, CLOUD]",Global/Remote Rate,2026-09-12 18:46:21.363750
3,4,Senior Communications Officer Strategic Commun...,[],Global/Remote Rate,2026-09-12 18:46:21.365241
4,5,Senior People & Talent Operations Partner,[],Global/Remote Rate,2026-09-12 18:46:21.366481
...,...,...,...,...,...
117,118,DevOps & Cloud Infrastructure Engineer,"[DOCKER, TERRAFORM, GITHUB ACTIONS, LINUX, AWS...","RM,",2026-09-20 00:15:08.087505
118,119,Software Engineer (Software Engineer - Indeed ...,"[SPRING BOOT, SQL, GIT, JAVA, PYTHON]","RM 5,500 - RM 9,000",2026-09-20 00:15:08.242203
119,120,Cloud & Systems Administrator (Indeed Fallback),"[DOCKER, TERRAFORM, LINUX, CI/CD, AWS, KUBERNE...","RM,",2026-09-20 00:15:08.248578
120,121,Full Stack Web Developer (Indeed Fallback),"[JAVASCRIPT, TYPESCRIPT, POSTGRESQL, NODE.JS, ...","RM 4,800 - RM 7,800",2026-09-20 00:15:08.255609
